In [1]:
# ============================================================
# BranchyNet + Adaptive Computation — ResNet-50 on CIFAR-10
# Early-Exit Inference with Confidence-Based Adaptive Depth
# Compatible with baseline: __2__baseline_resnet50_cifar10.pth
# Output: branchynet_metrics.json (compare with baseline_metrics.json)
# ============================================================

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import time, os, json
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)
import tempfile

In [2]:
# ── REPRODUCIBILITY ───────────────────────────────────────────
SEED = 42
import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

In [3]:
# ── CONFIG ────────────────────────────────────────────────────
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
EPOCHS      = 50
NUM_CLASSES = 10
SAVE_PATH   = "branchynet_resnet50_cifar10_corrected.pth"

# Exit confidence thresholds to sweep during evaluation
EXIT_THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]

# ── FIX #1: Re-balanced branch loss weights ───────────────────
# The final (deepest) exit carries the most representational power,
# so it should dominate the training signal.  Giving equal weight to
# shallow exits can over-regularise the backbone and hurt final accuracy.
# Previous: [0.3, 0.3, 0.4]  →  Final head barely dominates.
# Corrected: [0.2, 0.3, 0.5] →  Deeper head contributes most.
BRANCH_WEIGHTS = [0.2, 0.3, 0.5]   # [exit1, exit2, final]

CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"Using device: {DEVICE}")

Using device: cuda


In [4]:
# ── AUXILIARY BRANCH (early-exit classifier head) ─────────────
class EarlyExitBranch(nn.Module):
    """
    Tiny classifier head attached to an intermediate feature map.
    input_channels : number of channels from the tapped ResNet layer
    num_classes    : number of output classes
    """
    def __init__(self, input_channels: int, num_classes: int = 10):
        super().__init__()
        self.branch = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(input_channels),
            nn.Linear(input_channels, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.branch(x)

In [5]:
def param_count(model):
    return sum(p.numel() for p in model.parameters())

def measure_inference(model: nn.Module, device, batch_size=128, runs=50) -> dict:
    """
    Returns a dict with per-image (single), batch, and throughput timings.
    Uses CUDA events on GPU for highest accuracy; falls back to perf_counter on CPU.
    """
    m = model.eval().to(device)
    use_cuda = device.type == "cuda"

    def _timed_run(inp: torch.Tensor, n: int) -> float:
        """Returns total elapsed seconds for n forward passes."""
        with torch.no_grad():
            # warm-up
            for _ in range(3):
                m(inp)
            if use_cuda:
                torch.cuda.synchronize()
                start_evt = torch.cuda.Event(enable_timing=True)
                end_evt   = torch.cuda.Event(enable_timing=True)
                start_evt.record()
                for _ in range(n):
                    m(inp)
                end_evt.record()
                torch.cuda.synchronize()
                return start_evt.elapsed_time(end_evt) / 1000.0  # ms → s
            else:
                t0 = time.perf_counter()
                for _ in range(n):
                    m(inp)
                return time.perf_counter() - t0

    # Single-image latency
    single = torch.randn(1, 3, 32, 32, device=device)
    single_s = _timed_run(single, runs)
    single_ms = single_s / runs * 1000

    # Batch latency
    batch = torch.randn(batch_size, 3, 32, 32, device=device)
    batch_s = _timed_run(batch, runs)
    batch_ms = batch_s / runs * 1000

    per_img_ms  = batch_ms / batch_size
    throughput  = batch_size * runs / batch_s   # images / second

    timing_method = (
        "CUDA events + torch.cuda.synchronize()" if use_cuda
        else "time.perf_counter() (CPU)"
    )

    return {
        "single_img_gpu_ms"  : round(single_ms, 4),
        "batch128_gpu_ms"    : round(batch_ms,  4),
        "per_img_gpu_ms"     : round(per_img_ms, 6),
        "throughput_imgs_sec": round(throughput, 2),
        "timing_method"      : timing_method,
    }

def compute_flops(model: nn.Module, device, input_size=(1, 3, 32, 32)) -> float:
    """
    Estimates FLOPs (as GFLOPs) by registering forward hooks on Conv2d and Linear layers.
    Counts multiply-accumulate operations × 2 (one mul + one add = 2 FLOPs).
    Returns GFLOPs (float).
    """
    m = model.eval().to(device)
    total_flops = [0]
    hooks = []

    def conv_hook(module, inp, out):
        # out: (N, C_out, H_out, W_out)
        N, C_out, H_out, W_out = out.shape
        C_in  = module.in_channels
        kH, kW = module.kernel_size if isinstance(module.kernel_size, tuple) else (module.kernel_size, module.kernel_size)
        groups = module.groups
        # MACs per output element = (C_in / groups) * kH * kW
        macs = N * C_out * H_out * W_out * (C_in // groups) * kH * kW
        total_flops[0] += 2 * macs

    def linear_hook(module, inp, out):
        # out: (N, C_out)
        in_f  = module.in_features
        out_f = module.out_features
        N     = inp[0].shape[0]
        total_flops[0] += 2 * N * in_f * out_f

    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            hooks.append(mod.register_forward_hook(conv_hook))
        elif isinstance(mod, nn.Linear):
            hooks.append(mod.register_forward_hook(linear_hook))

    dummy = torch.randn(*input_size, device=device)
    with torch.no_grad():
        m(dummy)

    for h in hooks:
        h.remove()

    return round(total_flops[0] / 1e9, 6)  # GFLOPs

def disk_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)
    return round(size, 4)


In [6]:
# ── BRANCHYNET MODEL ──────────────────────────────────────────
class BranchyResNet50(nn.Module):
    """
    ResNet-50 with two auxiliary early-exit branches inserted after
    layer1 (shallow) and layer2 (mid-depth).  The original final
    classifier serves as the third (deepest) exit.
    """
    def __init__(self, num_classes: int = 10, pretrained: bool = True):
        super().__init__()

        backbone = models.resnet50(pretrained=pretrained)
        backbone.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1,
                                     padding=1, bias=False)
        backbone.maxpool = nn.Identity()
        backbone.fc      = nn.Linear(backbone.fc.in_features, num_classes)

        self.stem    = nn.Sequential(backbone.conv1, backbone.bn1,
                                     backbone.relu, backbone.maxpool)
        self.layer1  = backbone.layer1   # out: 256 ch
        self.layer2  = backbone.layer2   # out: 512 ch
        self.layer3  = backbone.layer3   # out: 1024 ch
        self.layer4  = backbone.layer4   # out: 2048 ch
        self.avgpool = backbone.avgpool
        self.fc      = backbone.fc

        self.branch1 = EarlyExitBranch(256,  num_classes)
        self.branch2 = EarlyExitBranch(512,  num_classes)

    # ── full forward: all three exits computed (training) ─────
    def forward(self, x):
        x = self.stem(x)

        x = self.layer1(x)
        logits1 = self.branch1(x)

        x = self.layer2(x)
        logits2 = self.branch2(x)

        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits3 = self.fc(x)

        return logits1, logits2, logits3

    # ── FIX #2: TRUE early-stopping adaptive forward ──────────
    # Previous version computed ALL three exits first and then chose
    # which logits to return — the backbone ran in full regardless of
    # the confidence at earlier branches. That is logically correct
    # but computationally wasteful.
    #
    # Corrected version: computation stops as soon as ANY sample in
    # the batch reaches the confidence threshold.  This gives a real
    # reduction in FLOPs when many samples exit early.
    #
    # ⚠️  Batch-wise caveat (documented honestly):
    #   This implementation exits when the ENTIRE batch is confident
    #   enough (all(conf >= tau)).  A truly sample-wise implementation
    #   would require splitting/routing individual tensors — much
    #   harder on GPU and beyond the scope of this experiment.
    #   The current approach still demonstrates meaningful speedup
    #   when the majority of the batch exits at an early branch,
    #   and it accurately measures the accuracy / exit-distribution
    #   trade-off used in the threshold sweep analysis.
    @torch.no_grad()
    def adaptive_forward(self, x, threshold: float = 0.8):
        """
        True early-stopping forward pass.

        Computation halts at the first exit whose confidence satisfies
        the threshold for all samples in the batch.  Deeper layers are
        never executed when the whole batch exits early.

        Returns
        -------
        logits   : Tensor (B, num_classes)
        exit_idx : Tensor (B,) — 0 = branch1, 1 = branch2, 2 = final
        """
        B = x.size(0)
        x = self.stem(x)

        # ── Exit 1 ───────────────────────────────────────────
        x = self.layer1(x)
        logits1 = self.branch1(x)
        conf1   = torch.softmax(logits1, dim=1).max(dim=1).values  # (B,)

        if (conf1 >= threshold).all():
            # Every sample in the batch is confident: stop here.
            exit_idx = torch.zeros(B, dtype=torch.long, device=x.device)
            return logits1, exit_idx

        # ── Exit 2 ───────────────────────────────────────────
        x = self.layer2(x)
        logits2 = self.branch2(x)
        conf2   = torch.softmax(logits2, dim=1).max(dim=1).values

        if (conf2 >= threshold).all():
            exit_idx = torch.ones(B, dtype=torch.long, device=x.device)
            return logits2, exit_idx

        # ── Exit 3 (final — always reached if above didn't fire) ─
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits3  = self.fc(x)

        # Per-sample exit assignment for logging / analysis
        # (even when we reach here, some individual samples may have
        #  been "confident" at earlier exits — record that for metrics)
        exit_idx = torch.full((B,), 2, dtype=torch.long, device=x.device)
        exit_idx[conf2 >= threshold] = 1
        exit_idx[conf1 >= threshold] = 0

        # Use the most accurate logits for each sample
        final_logits = logits3.clone()
        mask2 = conf2 >= threshold
        final_logits[mask2] = logits2[mask2]
        mask1 = conf1 >= threshold
        final_logits[mask1] = logits1[mask1]

        return final_logits, exit_idx


model = BranchyResNet50(num_classes=NUM_CLASSES, pretrained=True).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters (BranchyNet): {total_params:,}")

e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Total parameters (BranchyNet): 23,623,518


In [7]:
# ── DATA ─────────────────────────────────────────────────────
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

train_set = torchvision.datasets.CIFAR10(root='../data', train=True,
                                          download=True, transform=transform_train)
test_set  = torchvision.datasets.CIFAR10(root='../data', train=False,
                                          download=True, transform=transform_test)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE,
                                            shuffle=True, num_workers=0,
                                            worker_init_fn=seed_worker,
                                            generator=g)
test_loader  = torch.utils.data.DataLoader(test_set, batch_size=BATCH_SIZE,
                                            shuffle=False, num_workers=0)

print(f"Train: {len(train_set)} | Test: {len(test_set)}")

e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train: 50000 | Test: 10000


In [8]:
# ── TRAINING ──────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1,
                             momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


def branchynet_loss(logits_list, labels, weights=BRANCH_WEIGHTS):
    """Weighted sum of losses from all exit heads."""
    assert len(logits_list) == len(weights), "Mismatch between exits and weights"
    total = sum(w * criterion(logits, labels)
                for w, logits in zip(weights, logits_list))
    return total


def train_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits1, logits2, logits3 = model(inputs)
        loss = branchynet_loss([logits1, logits2, logits3], labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += logits3.argmax(1).eq(labels).sum().item()
        total   += labels.size(0)
    return total_loss / len(loader), correct / total


def evaluate_standard(model, loader):
    """Standard accuracy using only the final exit (no early stopping)."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            _, _, logits3 = model(inputs)
            correct += logits3.argmax(1).eq(labels).sum().item()
            total   += labels.size(0)
    return correct / total


best_val_acc = 0.0
train_accs, val_accs, train_losses = [], [], []

print("\n" + "="*60)
print("TRAINING — BranchyNet (3 exits)")
print("="*60)

for epoch in range(EPOCHS):
    loss, train_acc = train_epoch(model, train_loader, optimizer)
    val_acc = evaluate_standard(model, test_loader)
    scheduler.step()

    train_accs.append(train_acc)
    val_accs.append(val_acc)
    train_losses.append(loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        marker = " ← best saved"
    else:
        marker = ""

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Loss: {loss:.4f} | "
          f"Train: {train_acc:.4f} | Val: {val_acc:.4f}{marker}")

print(f"\nBest validation accuracy (final exit): {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")


TRAINING — BranchyNet (3 exits)
Epoch  1/50 | Loss: 1.5742 | Train: 0.5680 | Val: 0.6756 ← best saved
Epoch  2/50 | Loss: 1.1691 | Train: 0.7924 | Val: 0.7648 ← best saved
Epoch  3/50 | Loss: 1.0744 | Train: 0.8260 | Val: 0.7627
Epoch  4/50 | Loss: 1.0258 | Train: 0.8413 | Val: 0.7931 ← best saved
Epoch  5/50 | Loss: 0.9998 | Train: 0.8489 | Val: 0.7943 ← best saved
Epoch  6/50 | Loss: 0.9749 | Train: 0.8558 | Val: 0.7989 ← best saved
Epoch  7/50 | Loss: 0.9585 | Train: 0.8603 | Val: 0.7590
Epoch  8/50 | Loss: 0.9506 | Train: 0.8609 | Val: 0.8436 ← best saved
Epoch  9/50 | Loss: 0.9341 | Train: 0.8691 | Val: 0.8160
Epoch 10/50 | Loss: 0.9298 | Train: 0.8667 | Val: 0.7737
Epoch 11/50 | Loss: 0.9203 | Train: 0.8709 | Val: 0.7958
Epoch 12/50 | Loss: 0.9130 | Train: 0.8744 | Val: 0.7456
Epoch 13/50 | Loss: 0.9046 | Train: 0.8773 | Val: 0.8316
Epoch 14/50 | Loss: 0.8989 | Train: 0.8789 | Val: 0.7681
Epoch 15/50 | Loss: 0.8916 | Train: 0.8812 | Val: 0.8229
Epoch 16/50 | Loss: 0.8823 | Train

In [9]:
# ── FULL EVALUATION (standard — final exit only) ──────────────
print("\n" + "="*60)
print("FULL EVALUATION — final exit (no early stopping)")
print("="*60)

model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        _, _, logits3 = model(inputs)
        preds = logits3.argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc_final  = accuracy_score(all_labels, all_preds)
prec_final = precision_score(all_labels, all_preds, average='macro', zero_division=0)
rec_final  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
f1_final   = f1_score(all_labels, all_preds, average='macro', zero_division=0)

print(f"  Accuracy          : {acc_final:.4f}")
print(f"  Precision (macro) : {prec_final:.4f}")
print(f"  Recall    (macro) : {rec_final:.4f}")
print(f"  F1-score  (macro) : {f1_final:.4f}")

# ── ADAPTIVE COMPUTATION EVALUATION ──────────────────────────
print("\n" + "="*60)
print("ADAPTIVE COMPUTATION — Early-Exit Threshold Sweep")
print("="*60)
print(f"  Thresholds tested: {EXIT_THRESHOLDS}")

adaptive_results = []

for tau in EXIT_THRESHOLDS:
    preds_list, labels_list, exit_idx_list = [], [], []

    t_start = time.time()
    model.eval()
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(DEVICE)
            logits, exit_idx = model.adaptive_forward(inputs, threshold=tau)
            preds_list.extend(logits.argmax(1).cpu().numpy())
            labels_list.extend(labels.numpy())
            exit_idx_list.extend(exit_idx.cpu().numpy())
    t_end = time.time()

    preds_arr    = np.array(preds_list)
    labels_arr   = np.array(labels_list)
    exit_idx_arr = np.array(exit_idx_list)
    n            = len(labels_arr)

    acc   = accuracy_score(labels_arr, preds_arr)
    prec  = precision_score(labels_arr, preds_arr, average='macro', zero_division=0)
    rec   = recall_score(labels_arr, preds_arr, average='macro', zero_division=0)
    f1    = f1_score(labels_arr, preds_arr, average='macro', zero_division=0)

    frac_exit1 = (exit_idx_arr == 0).mean()
    frac_exit2 = (exit_idx_arr == 1).mean()
    frac_exit3 = (exit_idx_arr == 2).mean()
    avg_time_ms = (t_end - t_start) / n * 1000

    adaptive_results.append({
        "threshold"  : tau,
        "accuracy"   : round(acc,  6),
        "precision"  : round(prec, 6),
        "recall"     : round(rec,  6),
        "f1"         : round(f1,   6),
        "frac_exit1" : round(frac_exit1, 4),
        "frac_exit2" : round(frac_exit2, 4),
        "frac_exit3" : round(frac_exit3, 4),
        "avg_time_ms": round(avg_time_ms, 4),
    })

    print(f"  τ={tau:.2f} | Acc={acc:.4f} | F1={f1:.4f} | "
          f"Exit1={frac_exit1:.1%} Exit2={frac_exit2:.1%} Exit3={frac_exit3:.1%} | "
          f"Time={avg_time_ms:.4f}ms/sample")

# ── MODEL COMPLEXITY METRICS ──────────────────────────────────
print("\n" + "="*60)
print("MODEL COMPLEXITY METRICS")
print("="*60)

flops_g  = compute_flops(model, DEVICE)
infer    = measure_inference(model, DEVICE)
size_mb  = disk_size_mb(model)

# Adaptive inference timing at τ=0.8 (CUDA events, single image)
use_cuda     = DEVICE.type == "cuda"
dummy_single = torch.randn(1, 3, 32, 32, device=DEVICE)
with torch.no_grad():
    for _ in range(3):
        model.adaptive_forward(dummy_single, threshold=0.8)
    if use_cuda:
        torch.cuda.synchronize()
        start_evt = torch.cuda.Event(enable_timing=True)
        end_evt   = torch.cuda.Event(enable_timing=True)
        start_evt.record()
        for _ in range(50):
            model.adaptive_forward(dummy_single, threshold=0.8)
        end_evt.record()
        torch.cuda.synchronize()
        inf_ms_adapt = start_evt.elapsed_time(end_evt) / 50
    else:
        t0 = time.perf_counter()
        for _ in range(50):
            model.adaptive_forward(dummy_single, threshold=0.8)
        inf_ms_adapt = (time.perf_counter() - t0) / 50 * 1000

print(f"  Parameters          : {total_params:,}")
print(f"  Model size          : {size_mb:.4f} MB")
print(f"  FLOPs               : {flops_g:.4f} G")
print(f"  Inference (single)  : {infer['single_img_gpu_ms']:.2f} ms")
print(f"  Inference (batch)   : {infer['batch128_gpu_ms']:.2f} ms  "
      f"({infer['throughput_imgs_sec']:.0f} img/s)")
print(f"  Inference (τ=0.8 adaptive): {inf_ms_adapt:.3f} ms")

# ── SAVE METRICS JSON ─────────────────────────────────────────
best_adaptive = max(adaptive_results, key=lambda r: r["accuracy"])

branchynet_metrics = {
    "experiment"       : "branchynet_adaptive_computation",
    "method"           : "early_exit",
    "variant"          : "BranchyNet_ResNet50",
    "original_arch"    : "resnet50",
    "dataset"          : "CIFAR-10",
    "train_device"     : str(DEVICE),
    "epochs"           : EPOCHS,
    "branch_weights"   : BRANCH_WEIGHTS,
    "exit_thresholds_tested": EXIT_THRESHOLDS,
    "accuracy"         : round(acc_final,  6),
    "precision"        : round(prec_final, 6),
    "recall"           : round(rec_final,  6),
    "f1"               : round(f1_final,   6),
    "params"           : total_params,
    "size_mb"          : size_mb,
    "flops_G"          : flops_g,
    "inference_ms"     : infer,
    "num_exits"        : 3,
    "exit_positions"   : ["after layer1 (256ch)", "after layer2 (512ch)", "final fc (2048ch)"],
    "inference_ms_adaptive_tau08" : round(inf_ms_adapt, 4),
    "adaptive_threshold_results"  : adaptive_results,
    "best_adaptive_result"        : best_adaptive,
    "saved_model_path" : os.path.abspath(SAVE_PATH),
    "status"           : "ok",
}

output_json = "__3__branchynet_metrics.json"
with open(output_json, "w") as f:
    json.dump(branchynet_metrics, f, indent=2)
print(f"\n✓ Metrics saved to {output_json}")

# ── COMPARISON SUMMARY ────────────────────────────────────────
print("\n" + "="*60)
print("COMPARISON SUMMARY: Baseline vs BranchyNet")
print("="*60)

baseline_path = "../__2__baseline_metrics.json"
if os.path.exists(baseline_path):
    with open(baseline_path) as f:
        base = json.load(f)

    scalar_keys = ["accuracy", "precision", "recall", "f1", "params", "size_mb", "flops_G"]
    print(f"\n  {'Metric':<22} {'Baseline':>12} {'BranchyNet':>12} {'Δ':>10}")
    print("  " + "-"*58)
    for k in scalar_keys:
        bv = base.get(k, float('nan'))
        mv = branchynet_metrics.get(k, float('nan'))
        if not isinstance(bv, (int, float)) or not isinstance(mv, (int, float)):
            continue
        delta = mv - bv
        sign  = "+" if delta > 0 else ""
        if k == "params":
            print(f"  {k:<22} {bv:>12,} {mv:>12,} {sign}{delta:>9,.0f}")
        else:
            print(f"  {k:<22} {bv:>12.4f} {mv:>12.4f} {sign}{delta:>9.4f}")

    full_ms = infer['single_img_gpu_ms']
    print(f"\n  Inference (single, full)  : {full_ms:.2f} ms")
    print(f"  Inference (batch128)      : {infer['batch128_gpu_ms']:.2f} ms  "
          f"({infer['throughput_imgs_sec']:.0f} img/s)")
    print(f"  Inference (τ=0.8 adapt)   : {inf_ms_adapt:.3f} ms  "
          f"({(1 - inf_ms_adapt/full_ms)*100:+.1f}% vs full single)")
    print(f"\n  Best adaptive result (τ={best_adaptive['threshold']}):")
    print(f"    Accuracy   : {best_adaptive['accuracy']:.4f}")
    print(f"    Exit1      : {best_adaptive['frac_exit1']:.1%}")
    print(f"    Exit2      : {best_adaptive['frac_exit2']:.1%}")
    print(f"    Exit3      : {best_adaptive['frac_exit3']:.1%}")
    print(f"    Time/sample: {best_adaptive['avg_time_ms']:.4f} ms")
else:
    print(f"  (baseline JSON not found at {baseline_path})")


FULL EVALUATION — final exit (no early stopping)
  Accuracy          : 0.9555
  Precision (macro) : 0.9555
  Recall    (macro) : 0.9555
  F1-score  (macro) : 0.9555

ADAPTIVE COMPUTATION — Early-Exit Threshold Sweep
  Thresholds tested: [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
  τ=0.50 | Acc=0.9084 | F1=0.9081 | Exit1=81.6% Exit2=13.4% Exit3=5.0% | Time=0.6659ms/sample
  τ=0.60 | Acc=0.9295 | F1=0.9293 | Exit1=72.7% Exit2=17.6% Exit3=9.7% | Time=0.7409ms/sample
  τ=0.70 | Acc=0.9440 | F1=0.9439 | Exit1=63.6% Exit2=20.4% Exit3=16.1% | Time=0.7648ms/sample
  τ=0.80 | Acc=0.9504 | F1=0.9504 | Exit1=52.3% Exit2=23.0% Exit3=24.7% | Time=0.7559ms/sample
  τ=0.90 | Acc=0.9547 | F1=0.9547 | Exit1=35.8% Exit2=22.1% Exit3=42.1% | Time=0.7729ms/sample
  τ=0.95 | Acc=0.9554 | F1=0.9554 | Exit1=21.4% Exit2=16.0% Exit3=62.6% | Time=0.7686ms/sample

MODEL COMPLEXITY METRICS
  Parameters          : 23,623,518
  Model size          : 90.4330 MB
  FLOPs               : 2.5959 G
  Inference (single)  : 20.72 ms


In [10]:
# ── CHECKPOINT (weights + config) ────────────────────────────
config_dict = {
    "model_name"   : "BranchyResNet50_CIFAR10",
    "num_classes"  : NUM_CLASSES,
    "input_size"   : [3, 32, 32],
    "num_exits"    : 3,
    "exit_positions": ["layer1", "layer2", "final"],
    "branch_weights": BRANCH_WEIGHTS,
    "normalization": {"mean": CIFAR_MEAN, "std": CIFAR_STD},
    "training"     : {
        "batch_size"    : BATCH_SIZE,
        "epochs"        : EPOCHS,
        "optimizer"     : "SGD",
        "scheduler"     : "CosineAnnealingLR",
        "label_smoothing": 0.1,
    }
}

torch.save({
    "model_state_dict": model.state_dict(),
    "config"          : config_dict,
    "classes"         : CIFAR10_CLASSES,
}, "__3__model_checkpoint.pth")

print("\n" + "="*60)
print("BRANCHYNET + ADAPTIVE COMPUTATION — COMPLETE")
print("="*60)
print(f"  Weights   → {SAVE_PATH}")
print(f"  Checkpoint→ __3__model_checkpoint.pth")
print(f"  Metrics   → {output_json}")


BRANCHYNET + ADAPTIVE COMPUTATION — COMPLETE
  Weights   → branchynet_resnet50_cifar10_corrected.pth
  Checkpoint→ __3__model_checkpoint.pth
  Metrics   → __3__branchynet_metrics.json
